# ML-05 · Образовательная миграция ВО и СПО

**«Демографический советник» · Этап II · `stage2-ml5-v2.0.0`**

Единый самостоятельный Colab-ноутбук. Источники — **только** неизменённый `handoff/07_v2/dataset_index.json` приватного репозитория Этапа I, его штатные исходные таблицы и источниковые манифесты. Другие ML-исследования не читаются, не импортируются и не изменяются.

**Семь проверок:** (1) потоки между регионами; (2) приём в учреждения; (3) образовательный отток; (4) программы; (5) трудоустройство; (6) межрегиональная сеть; (7) муниципальная демография: E0 против E1. При нехватке данных применяется `NOT_FEASIBLE`, не создаются синтетические потоки, география трудоустройства или фиктивные нули.

**Статус выдаваемого файла:** разработан и статически проверен; **не** является исполненным научным исследованием. `SMOKE` проверяет воспроизводимую загрузку/код, `FULL` исполняет научный протокол. Публикация запрещена до `FULL` и успешных проверок всех применимых компонентов.

**Инструкция:** откройте этот файл в Colab, укажите токен с правом чтения приватного репозитория в Colab Secrets как `GITHUB_TOKEN`, при полном прогоне установите в следующей ячейке `MODE="FULL"`, запустите **Runtime → Run all**. Google Drive не требуется.

## 1. Паспорт, гипотезы и правило научной приёмки

Образовательная мобильность означает измеренное различие между территорией предыдущего образования и территорией обучения; она **не равна** постоянному миграционному учёту. `origin_residence`, `origin_previous_education`, `destination_study`, `employment_location` и `postgraduation_residence` считаются разными величинами. Данные о трудоустройстве и заработке не доказывают закрепление выпускника. Учреждения и филиалы сохраняют собственные географические ключи; юридический адрес головной организации не приписывается филиалу.

H5.1 — модель улучшает прогноз подтверждённых OD-потоков относительно инерции; H5.2 — характеристики организации помогают прогнозировать приём; H5.3 — образовательный отток неоднороден; H5.4 — программы различаются по географии выбора; H5.5 — характеристики связаны с трудоустройством; H5.6 — регионы образуют устойчивые сетевые центры; H5.7 — образовательная инфраструктура улучшает муниципальный прогноз внеэкономических факторов. **Ни одна гипотеза не считается подтверждённой до независимого теста.**

## 2. Единая конфигурация

Правьте **только эту ячейку**. `SMOKE` не публикует результаты и не приравнивается к научному прогону; `FULL` не публикует результаты, если не выполнен полный научный QA. Идентификаторы показателей выбираются **только по их точным оригинальным названиям** после просмотра автоматически выгруженного словаря возможных показателей; неоднозначные названия не выбираются догадкой.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class Config:
    MODE: str = "SMOKE"                        # "SMOKE" или "FULL"
    REPO: str = "AntonDozhdikov/Demograf_sovetnik"
    INPUT_REF: str = "main"
    INDEX: str = "handoff/07_v2/dataset_index.json"
    LOADER_REFERENCE: str = "handoff/07_v2/stage2_loader.py"
    SEGMENT: str = "stage2/ml05_education"
    VERSION: str = "stage2-ml5-v2.0.0"
    TARGET_BRANCH_PREFIX: str = "ml05-education"
    ROOT: str = "/content/ml05_education"
    PUBLISH_AFTER_FULL_QA: bool = True
    HORIZONS: tuple = (1,)
    DEV_SEEDS: tuple = (17, 29, 41, 53, 67, 79, 83, 97, 101, 113)
    FINAL_SEEDS: tuple = tuple(range(1001,1051))
    MAX_FINAL_SEEDS_SMOKE: int = 1
    BOOTSTRAP_REPS: int = 400
    MIN_TRAIN_ROWS: int = 80
    MIN_TEST_ROWS: int = 30
    MIN_YEARS: int = 4
    MIN_MUNI_E1_SUPPORT: int = 60
    PUBLICATION_LAG_YEARS: int = 1
    MAX_ASSET_MB: int = 900
    SMOKE_ASSET_LIMIT: int = 4
    # Заполняются после содержательной проверки справочника indicator_name.
    APPROVED_E02_ADMISSIONS: tuple = ()          # точные названия E02 индикаторов приёма
    APPROVED_SPO_ADMISSIONS: tuple = ()          # точные названия СПО индикаторов приёма
    APPROVED_VPO_ADMISSIONS: tuple = ()          # точные названия ВПО-1 приёма, если обоснованы
    ALLOW_CONDITIONAL_OD_SEMANTICS: bool = False # если не определено основание происхождения, OD только как условная мобильность
    NETWORK_MIN_YEARS: int = 2

CFG = Config()
assert CFG.MODE in {"SMOKE", "FULL"}
OUT = Path(CFG.ROOT)
OUT.mkdir(parents=True,exist_ok=True)
(OUT/"inputs").mkdir(exist_ok=True)
(OUT/"derived").mkdir(exist_ok=True)
(OUT/"figures").mkdir(exist_ok=True)
print(f"ML-05 · режим {CFG.MODE} · версия {CFG.VERSION} · сегмент {CFG.SEGMENT}")

## 3–4. Диагностика Colab и минимальные зависимости

GPU для базовых методов не требуется. Установки выполняются только для отсутствующих пакетов; сообщения pip и предупреждения о версиях не являются доказательством корректности расчётов.

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil, time, hashlib, json, traceback, re
from datetime import datetime, timezone
CORE={"numpy":"numpy", "pandas":"pandas", "pyarrow":"pyarrow", "duckdb":"duckdb", "sklearn":"scikit-learn", "scipy":"scipy", "catboost":"catboost", "networkx":"networkx", "matplotlib":"matplotlib", "xlsxwriter":"XlsxWriter", "nbformat":"nbformat", "requests":"requests", "tqdm":"tqdm"}
missing=[p for m,p in CORE.items() if importlib.util.find_spec(m) is None]
if missing:
    proc=subprocess.run([sys.executable,"-m","pip","install","-q",*missing],capture_output=True,text=True)
    if proc.returncode: raise RuntimeError("Ошибка установки библиотек: "+", ".join(missing))
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, duckdb, requests, nbformat, networkx as nx
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
from tqdm.auto import tqdm
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
RUN_STARTED=time.monotonic()
DISK=shutil.disk_usage(CFG.ROOT)
print({"python":sys.version.split()[0],"cpu":os.cpu_count(),"free_gb":round(DISK.free/1e9,2),"ram_gb":round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1e9,2) if hasattr(os,'sysconf') else None,"gpu_optional":True})

## 5. Защищённое чтение handoff Этапа I

Аутентификация: Colab Secrets (`GITHUB_TOKEN`) или переменная окружения. Токен не записывается в выходные файлы. Для `git`- и `release`-объектов применяется ровно транспорт, заданный их `locator` в индексе; каждый скачанный файл проверяется по SHA-256. До успешного научного прогона записи в GitHub нет.

In [ ]:
import base64, getpass
from urllib.parse import quote

def get_token():
    token=os.environ.get("GITHUB_TOKEN","").strip()
    if not token:
        try:
            from google.colab import userdata
            token=(userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception: pass
    if not token: token=getpass.getpass("GitHub token (ввод скрыт): ").strip()
    if not token: raise RuntimeError("BLOCKED_AUTH: необходим GITHUB_TOKEN для приватного Stage I")
    return token
TOKEN=get_token()
API="https://api.github.com/repos/"+CFG.REPO
SESSION=requests.Session()
SESSION.headers.update({"Authorization":"Bearer "+TOKEN,"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"})

def safe_request(method,url,**kwargs):
    for retry in range(4):
        try:
            r=SESSION.request(method,url,timeout=kwargs.pop("timeout",(30,600)),**kwargs)
            if r.status_code in (429,502,503,504):
                r.close(); time.sleep(2**retry);continue
            if r.status_code in (401,403,404):
                status=r.status_code;r.close()
                raise PermissionError(f"GitHub {status}: не найден файл или недостаточно прав к приватному репозиторию")
            r.raise_for_status()
            return r
        except PermissionError: raise
        except requests.RequestException:
            if retry==3: raise RuntimeError("Сетевая ошибка при получении данных Stage I") from None
            time.sleep(2**retry)
    raise RuntimeError("GitHub: исчерпаны повторы")

def git_text(path,ref=CFG.INPUT_REF):
    u=API+"/contents/"+quote(path,safe="/")
    r=safe_request("GET",u,params={"ref":ref});data=r.json();r.close()
    if not isinstance(data,dict) or not data.get("content"):
        raise RuntimeError("Недоступно содержимое файла из GitHub Contents: "+path)
    return base64.b64decode(data["content"]).decode("utf-8")

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""):h.update(chunk)
    return h.hexdigest()

def download_asset(a):
    loc=a["locator"]; target=OUT/"inputs"/(hashlib.sha256(a["asset_id"].encode()).hexdigest()[:20]+".parquet")
    if target.exists() and sha256(target)==a["sha256"]:return target
    target.unlink(missing_ok=True)
    if loc["type"]=="git":
        u=API+"/contents/"+quote(loc["path"],safe="/")
        r=safe_request("GET",u,params={"ref":CFG.INPUT_REF})
        payload=r.json();r.close()
        if payload.get("content"):
            target.write_bytes(base64.b64decode(payload["content"]))
        else:
            rr=safe_request("GET",u,params={"ref":CFG.INPUT_REF},headers={"Accept":"application/vnd.github.raw+json"},stream=True)
            with open(target,"wb") as f:
                for part in rr.iter_content(2*1024*1024):
                    if part:f.write(part)
            rr.close()
    elif loc["type"]=="release":
        u=API+"/releases/assets/"+str(loc["asset_id"])
        r=safe_request("GET",u,headers={"Accept":"application/octet-stream"},stream=True)
        if "json" in r.headers.get("content-type","").lower():
            detail=r.json();r.close()
            r=safe_request("GET",detail["browser_download_url"],stream=True)
        with open(target,"wb") as f:
            for part in r.iter_content(2*1024*1024):
                if part:f.write(part)
        r.close()
    else: raise RuntimeError("Неизвестный транспорт Stage I: "+str(loc["type"]))
    actual=sha256(target)
    if actual!=a["sha256"]:
        target.unlink(missing_ok=True)
        raise RuntimeError("SHA256 не совпадает: "+a["asset_id"])
    return target

INDEX=json.loads(git_text(CFG.INDEX))
assert INDEX.get("version")=="7.2", "Версия handoff отличается: проверьте схему"
assert INDEX.get("repo")==CFG.REPO
ASSETS=INDEX["assets"]
assert len(ASSETS)>=1 and all(a.get("sha256_verified") for a in ASSETS)
assert all(len(a.get("sha256", ""))==64 for a in ASSETS)
reference=git_text(CFG.LOADER_REFERENCE)
SOURCE_MANIFESTS={segment:json.loads(git_text(INDEX["source_manifests"][segment])) for segment in ["03","04","05"]}
assert "fetch_asset" in reference and "releases/assets" in reference, "Изменился референсный протокол"
print(f"PASS_INPUT_CATALOG: версия {INDEX['version']}; {len(ASSETS)} объектов; схема и способ загрузки подтверждены")

## 6. DATA_GATE: каталог, фактические словари и определения

Сначала сопоставляются реальные поля исходников; индикаторные строки без подтверждённого смысла не преобразуются в зачисленных, образовательные мигранты или закрепившихся. `edu_od_year` по E01 относится к СПО, но сама схема не сообщает, измеряет ли происхождение место предыдущего образования или регистрацию. Поэтому термин остаётся «источник E01», пока не прочитано определение первичного показателя. ВО не получает подставленную матрицу СПО.

In [ ]:
from collections import defaultdict
ASSET_MAP={a["asset_id"]:a for a in ASSETS}

def choose(segment=None,role=None,source=None,prefix=None):
    return [a for a in ASSETS if (segment is None or a["segment"]==segment)
            and (role is None or a.get("role")==role)
            and (source is None or a.get("source_id")==source)
            and (prefix is None or a["asset_id"].startswith(prefix))]

def exactly_one(candidates,label):
    if len(candidates)!=1:raise RuntimeError(f"DATA_GATE: {label}: найдено {len(candidates)} вместо 1")
    return candidates[0]

def asset_df(a,columns=None,where=None,limit=None):
    path=download_asset(a)
    fields=set(a.get("columns_manifest") or [])
    if columns and fields and not set(columns).issubset(fields):
        raise RuntimeError("Неверные поля "+a["asset_id"]+": "+str(set(columns)-fields))
    con=duckdb.connect(database=":memory:")
    try:
        if columns:
            # имена столбцов берутся только из подтверждённого манифеста
            cols=", ".join('"'+c.replace('"','""')+'"' for c in columns)
        else: cols="*"
        sql=f"SELECT {cols} FROM read_parquet(?)"
        if where: sql+=" WHERE "+where  # where передаётся только локальными фиксированными выражениями
        if limit:sql+=" LIMIT "+str(int(limit))
        return con.execute(sql,[str(path)]).df()
    finally:con.close()

candidates={
 "spo_od": choose("03",role="education_od_flows"),
 "spo_registry": choose("03",role="education_organizations"),
 "spo_org": sorted(choose("03",role="organization_indicators"),key=lambda a:a["rows_actual"],reverse=True),
 "spo_program": choose("03",role="regional_spo"),
 "vo_e02": choose("04",source="E02"),
 "vo_vpo_admission":choose("04",source="VPO_2_1_1"),
 "graduates":choose("05",role="organization_total"),
 "mun":choose("01",source="D01"),
 "spatial":choose("06",role="derived"),
}
ASSET_AUDIT=pd.DataFrame([{"asset_id":a["asset_id"],"segment":a["segment"],"role":a.get("role",""),"source":a["source_id"],"rows_catalog":a.get("rows_actual"),"sha256":a["sha256"],"fields":", ".join(a.get("columns_manifest",[])[:30])} for a in ASSETS])
STATUS={k:{"status":"PENDING","reason":"Данные пока не анализировались"} for k in [f"ML-5.{i}" for i in range(1,8)]}
ISSUES=[];PROVENANCE=[];PANELS={};QA=[];PREDICTIONS=[];METRICS=[];SPLITS=[];FIGURES=[];MODELS_META=[];MODEL_FILES={};ACTIVE_MODULE=None

def issue(module,reason,missing=""):
    ISSUES.append({"module":module,"status":"NOT_FEASIBLE","reason":reason,"required_fields":missing})
    STATUS[module]={"status":"NOT_FEASIBLE","reason":reason}

def gate(name,df,keys=None):
    if df is None or df.empty:
        QA.append({"table":name,"status":"EMPTY","rows":0});return False
    bad=0
    if keys:
        if not set(keys).issubset(df.columns):raise AssertionError(f"{name}: нет ключей {set(keys)-set(df.columns)}")
        bad=int(df.duplicated(keys).sum())
    QA.append({"table":name,"status":"PASS" if bad==0 else "DUPLICATE_KEYS","rows":len(df),"duplicates":bad,"years":df["year"].nunique() if "year" in df.columns else None})
    return bad==0

def add_provenance(df,a,panel,semantics):
    if df is not None and not df.empty:
        PROVENANCE.append({"panel":panel,"asset_id":a["asset_id"],"source_id":a["source_id"],"source_sha256":a["sha256"],"rows":len(df),"definition":semantics})

assert len(candidates['spo_od'])==1, 'Нет однозначной таблицы СПО OD'
print("DATA_GATE: источники по ролям:", {k:len(v) for k,v in candidates.items()})

## 7. Шесть нормализованных панелей без неявных JOIN

Таблицы сохраняют собственные ключи. В показателях `year` — год, заявленный источником; год публикации хранится отдельно. Отсутствие записи OD не означает ноль. Экономические и региональные величины не раздаются муниципалитетам через `region` как будто это наблюдения муниципалитета. При несовпадении идентификаторов выполняется карантин, а не нечёткое массовое объединение.

In [ ]:
def sql_asset(a,sql):
    path=download_asset(a)
    con=duckdb.connect(database=":memory:")
    try:return con.execute(sql,[str(path)]).df()
    finally:con.close()

def normyear(x):return pd.to_numeric(x,errors="coerce").astype("Int64")
def normcode(x):return x.astype("string").str.strip().str.replace(r"\.0$","",regex=True)
def numeric(x):return pd.to_numeric(x,errors="coerce")

# 7a. Наблюдаемая OD-сеть СПО — без добавления необнаруженных рёбер.
spo_od_asset=exactly_one(candidates["spo_od"],"SPО OD")
od=asset_df(spo_od_asset,["year","origin_region_code","destination_region_code","origin_region_name","destination_region_name","indicator_value"])
od=od.rename(columns={"indicator_value":"flow_count","origin_region_code":"origin_region","destination_region_code":"destination_region"})
od["year"]=normyear(od["year"])
for c in ("origin_region","destination_region"):od[c]=normcode(od[c])
od["flow_count"]=numeric(od["flow_count"])
od["sector"]="СПО";od["source_definition"]="E01: источник предыдущего образования/проживания уточняется по первоисточнику"
od=od.loc[od.year.notna()&od.origin_region.notna()&od.destination_region.notna()&od.flow_count.notna()].copy()
od=od.loc[(od.flow_count>=0)&np.isclose(od.flow_count,np.round(od.flow_count))].copy()
OD_KEYS=["origin_region","destination_region","year","sector"]
if not gate("edu_od_year",od,OD_KEYS):
    dup=od.loc[od.duplicated(OD_KEYS,keep=False)].copy()
    PANELS["edu_od_ambiguous"]=dup
    od=od.loc[~od.duplicated(OD_KEYS,keep=False)]
    ISSUES.append({"module":"ML-5.1","status":"WARN","reason":"Неоднозначные OD-дубли исключены, не суммированы","required_fields":"доказательство непересечения категорий"})
PANELS["edu_od_year"]=od
add_provenance(od,spo_od_asset,"edu_od_year","наблюдаемые OD-значения СПО; нулевые рёбра не достраиваются")

# 7b. Муниципальная демография D01. ОКТМО — строка.
a_mun=exactly_one(candidates["mun"],"D01 муниципалитеты")
mun=asset_df(a_mun,["oktmo","region","municipality","mun_type","year","population","migration","migration_rate"])
mun["oktmo"]=normcode(mun["oktmo"]);mun["year"]=normyear(mun["year"])
for c in ["population","migration","migration_rate"]:mun[c]=numeric(mun[c])
mun=mun.dropna(subset=["oktmo","year"]).copy()
if not gate("municipal_demography",mun,["oktmo","year"]):
    PANELS["municipal_duplicates"]=mun.loc[mun.duplicated(["oktmo","year"],keep=False)]
    mun=mun.loc[~mun.duplicated(["oktmo","year"],keep=False)]
PANELS["municipal_demography"]=mun
add_provenance(mun,a_mun,"municipal_demography","муниципальная демография D01, не отраслевое OD")

# 7c. Реестр ВО E02 с подтверждаемыми id и parent_id; медленно меняющиеся атрибуты.
if CFG.MODE=="FULL":
    a_e02=exactly_one(candidates["vo_e02"],"E02 ВО")
    q='SELECT DISTINCT "id", "parent_id", "name", "parent_name", "region", "oktmo", "year", "institution" FROM read_parquet(?) WHERE "id" IS NOT NULL AND "year" IS NOT NULL'
    vo_reg=sql_asset(a_e02,q)
    vo_reg["year"]=normyear(vo_reg.year)
    for c in ["id","parent_id","oktmo"]:vo_reg[c]=normcode(vo_reg[c])
    vo_reg["entity_id"]="ВО:"+vo_reg["id"].astype(str)
    vo_reg["group_id"]=np.where(vo_reg.parent_id.notna()&(vo_reg.parent_id!=""), "ВО:"+vo_reg.parent_id.astype(str),vo_reg.entity_id)
    vo_reg["sector"]="ВО"
    inconsistent=vo_reg.duplicated(["entity_id","year"],keep=False)
    PANELS["vo_registry_quarantine"]=vo_reg.loc[inconsistent].copy()
    vo_reg=vo_reg.loc[~inconsistent].copy()
    gate("vo_registry",vo_reg,["entity_id","year"])
    add_provenance(vo_reg,a_e02,"edu_org_registry_vo","реестр E02; точка расположения подтверждается отдельно")
else:vo_reg=pd.DataFrame()

# 7d. Реестр СПО: id и head сохраняются как исходные строки, тип связи не угадывается.
if CFG.MODE=="FULL":
    a_sporeg=exactly_one(candidates["spo_registry"],"реестр СПО")
    cols=["year","region_name","region_oktmo","name","city","lat","lon","type","id","address","head","branches"]
    spo_reg=asset_df(a_sporeg,cols).drop_duplicates()
    spo_reg["year"]=normyear(spo_reg.year)
    spo_reg["id"]=normcode(spo_reg["id"])
    spo_reg["entity_id"]="СПО:"+spo_reg["id"].fillna("НЕИЗВЕСТНО")
    spo_reg["group_id"]=spo_reg["entity_id"]
    spo_reg["sector"]="СПО";spo_reg["match_confidence"]="id_from_source_not_cross_registry"
    bad=spo_reg.duplicated(["entity_id","year"],keep=False)
    PANELS["spo_registry_quarantine"]=spo_reg.loc[bad].copy()
    spo_reg=spo_reg.loc[~bad].copy()
    gate("spo_registry",spo_reg,["entity_id","year"])
    add_provenance(spo_reg,a_sporeg,"edu_org_registry_spo","реестр организаций E01; адрес площадки не доказан для всех филиалов")
else:spo_reg=pd.DataFrame()
PANELS["edu_org_registry"]=pd.concat([vo_reg,spo_reg],ignore_index=True) if CFG.MODE=="FULL" else pd.DataFrame()

# 7e. Возможные индикаторы зачисления: словарь исходных формулировок, без выбора по regex.
if CFG.MODE=="FULL":
    a_spo_ind=candidates["spo_org"][0]
    spo_indicator_dictionary=sql_asset(a_spo_ind,'SELECT DISTINCT "indicator_name", "indicator_section", "indicator_unit" FROM read_parquet(?) WHERE "indicator_name" IS NOT NULL')
    e02_indicator_dictionary=sql_asset(a_e02,'SELECT DISTINCT "indicator_name", "indicator_unit" FROM read_parquet(?) WHERE "indicator_name" IS NOT NULL')
else:
    spo_indicator_dictionary=pd.DataFrame();e02_indicator_dictionary=pd.DataFrame()
PANELS["spo_indicator_dictionary"]=spo_indicator_dictionary
PANELS["e02_indicator_dictionary"]=e02_indicator_dictionary

def exact_indicators(actual,approved,label):
    approved=set(approved)
    if not approved:return False
    unknown=approved-set(actual.astype(str).tolist())
    if unknown:raise ValueError(f"{label}: нет точных индикаторов в первоисточнике: {sorted(unknown)}")
    return True

# 7f. Реальная панель учреждений×год только для однозначно выбранного показателя.
org_parts=[]
if CFG.MODE=="FULL" and exact_indicators(e02_indicator_dictionary.indicator_name,CFG.APPROVED_E02_ADMISSIONS,"E02"):
    chosen=CFG.APPROVED_E02_ADMISSIONS
    # Несколько строк одного индикатора (например, по специальностям) НЕ суммируем без доказательства разбиения.
    raw=asset_df(a_e02,["id","year","indicator_name","area","specialty_code","indicator_value"])
    raw=raw.loc[raw.indicator_name.isin(chosen)].copy()
    raw=raw.loc[raw.area.isna()&raw.specialty_code.isna()]
    raw["year"]=normyear(raw.year);raw["entity_id"]="ВО:"+normcode(raw.id).astype(str)
    raw["admissions"]=numeric(raw.indicator_value)
    raw=raw.dropna(subset=["admissions","year"])
    dup=raw.duplicated(["entity_id","year"],keep=False)
    PANELS["vo_admission_quarantine"]=raw.loc[dup]
    raw=raw.loc[~dup,["entity_id","year","admissions"]].copy()
    raw=raw.merge(vo_reg[["entity_id","year","group_id","region"]],on=["entity_id","year"],validate="one_to_one")
    raw["sector"]="ВО";org_parts.append(raw)
if CFG.MODE=="FULL" and exact_indicators(spo_indicator_dictionary.indicator_name,CFG.APPROVED_SPO_ADMISSIONS,"E01"):
    a=candidates["spo_org"][0]
    raw=asset_df(a,["name","region_name","year","indicator_name","org_value"])
    raw=raw.loc[raw.indicator_name.isin(CFG.APPROVED_SPO_ADMISSIONS)].copy()
    raw["year"]=normyear(raw.year);raw["admissions"]=numeric(raw.org_value)
    raw=raw.dropna(subset=["admissions","year"])
    # Сопоставление name×region×year только если 1:1 с устойчивым реестром.
    keys=["name","region_name","year"]
    lookup=spo_reg[keys+["entity_id","group_id"]].drop_duplicates()
    amb=lookup.duplicated(keys,keep=False)
    PANELS["spo_match_quarantine"]=lookup.loc[amb]
    lookup=lookup.loc[~amb]
    raw=raw.merge(lookup,on=keys,how="left",validate="many_to_one",indicator="_match")
    PANELS["spo_unmatched_indicators"]=raw.loc[raw._match!="both"].copy()
    raw=raw.loc[raw._match=="both"]
    dup=raw.duplicated(["entity_id","year"],keep=False)
    PANELS["spo_admission_quarantine"]=raw.loc[dup]
    raw=raw.loc[~dup,["entity_id","year","admissions","group_id","region_name"]].rename(columns={"region_name":"region"})
    raw["sector"]="СПО";org_parts.append(raw)
org_year=pd.concat(org_parts,ignore_index=True) if org_parts else pd.DataFrame()
PANELS["edu_org_year"]=org_year
gate("edu_org_year",org_year,["entity_id","year","sector"] if not org_year.empty else None)

# 7g. Программы: исходные коды без переприсваивания неизвестного crosswalk.
if CFG.MODE=="FULL":
    a_prog=exactly_one(candidates["spo_program"],"региональные программы СПО")
    program=asset_df(a_prog,["year","region_name","specialty_code","specialty_name","indicator_name","indicator_unit","region_value"])
    program["year"]=normyear(program.year);program["specialty_code"]=normcode(program.specialty_code)
    program["value"]=numeric(program.region_value)
    program=program.dropna(subset=["year","specialty_code","value"])
    program["sector"]="СПО";program["programme_family_unverified"]=program.specialty_code.str.slice(0,2)
    add_provenance(program,a_prog,"edu_program_year","названия программ и коды СПО в первичном классификаторе; crosswalk не утверждён")
else:program=pd.DataFrame()
PANELS["edu_program_year"]=program

# 7h. Выпускники: год в E04 не равен году наблюдения занятости.
if CFG.MODE=="FULL":
    a_grad=exactly_one(candidates["graduates"],"результаты выпускников")
    grad=asset_df(a_grad,["object_level","object_name","oktmo","gender","education_level","year","university","count_graduate","percent_employed","average_salary_fact_med"])
    grad["year"]=normyear(grad.year);grad["oktmo"]=normcode(grad.oktmo)
    for col in ["count_graduate","percent_employed","average_salary_fact_med"]:grad[col]=numeric(grad[col])
    grad["employment_reference_year"]=int(SOURCE_MANIFESTS["05"]["files"]["E04::organization_total"]["employment_reference"][:4])
    grad["measurement_definition"]="E04: employment_reference=2024-12-31; год исходной колонки year НЕ интерпретирован автоматически как год измерения"
    add_provenance(grad,a_grad,"graduate_outcomes","занятость и зарплата на дату референса манифеста 2024-12-31; без места работы")
else:grad=pd.DataFrame()
PANELS["graduate_outcomes"]=grad

print("Шесть логических панелей созданы/проверены:",{k:len(PANELS.get(k,[])) for k in ["edu_org_registry","edu_org_year","edu_od_year","edu_program_year","graduate_outcomes","municipal_demography"]})

## 8. Предрегистрация прогноза до просмотра тестовых ошибок

Единица прогноза — идентифицированная пара регионов или организация, год происхождения информации `origin_year`, горизонт `h=1`, конечный `target_year=origin_year+h`. Будущие значения не становятся признаками. Сравнение моделей только на совпадающих тестовых ключах. **Значения временного теста не используются ни для выбора модели, ни для подбора гиперпараметров**.

`ROLLING_TIME`: предпоследний год метки — development, последний — окончательный test; train — раньше. Если длины ряда/тестовой выборки не хватает, возвращается `NOT_FEASIBLE`. Простая интерпретация иерархических/групповых результатов — прогнозная, не причинная. Значение предыдущего года входит в baseline и модель; `t+1` не входит в матрицу признаков.

In [ ]:
EXPERIMENT_PLAN={
 "ML-5.1":{"target":"observed_spo_OD_count","base":"last observed flow","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"origin×destination","hypothesis":"H5.1"},
 "ML-5.2":{"target":"approved_annual_enrolment","base":"last institution value","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"institution/branch","hypothesis":"H5.2"},
 "ML-5.3":{"target":"observed_outflow_count","base":"previous outflow","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"region","hypothesis":"H5.3"},
 "ML-5.4":{"target":"programme enrolment only if verified","base":"previous programme value","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"region×programme","hypothesis":"H5.4"},
 "ML-5.5":{"target":"measured employment/wage if time available","base":"prior cohort observations","stats":"Ridge","ml":"CatBoost","cutoff":"employment measurement date","unit":"organization/cohort","hypothesis":"H5.5"},
 "ML-5.6":{"target":"observed network centrality/edges","base":"previous annual graph","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"region×year","hypothesis":"H5.6"},
 "ML-5.7":{"target":"municipal migration rate or change","base":"E0 prior migration and demographics","stats":"E0 Ridge","ml":"E1 CatBoost","cutoff":"origin_year","unit":"municipality","hypothesis":"H5.7"},
}
PLAN_HASH=hashlib.sha256(json.dumps({"experiment_plan":EXPERIMENT_PLAN,"config":asdict(CFG)},sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()

def make_temporal_panel(df,keys,target,features=None,h=1):
    assert h>=1
    if df.empty:return pd.DataFrame()
    keys=list(keys); required=keys+["year",target]
    if not set(required).issubset(df):raise KeyError("Нет целей/ключей: "+str(set(required)-set(df.columns)))
    if df.duplicated(keys+["year"]).any():raise ValueError("many-to-many в целевой панели")
    original=df.copy();original.year=normyear(original.year)
    original=original.dropna(subset=required)
    original[target]=numeric(original[target]);original=original.dropna(subset=[target])
    # Доступен только год y<=origin_year-publication_lag, а не значение origin_year.
    current=original[keys+["year",target]+(features or [])].rename(columns={target:"prior_value","year":"information_cutoff"})
    current["origin_year"]=(current.information_cutoff+CFG.PUBLICATION_LAG_YEARS).astype(int)
    future=original[keys+["year",target]].rename(columns={target:"actual","year":"target_year"})
    current["target_year"]=(current.origin_year+h).astype(int)
    # exact year-to-year join: нет искусственного перехода через пропущенные периоды
    panel=current.merge(future,on=keys+["target_year"],how="inner",validate="one_to_one")
    panel["entity_id"]=panel[keys].astype(str).agg("|".join,axis=1)
    panel["horizon"]=h
    assert (panel.information_cutoff<=panel.origin_year-CFG.PUBLICATION_LAG_YEARS).all()
    panel["source_vintage"]=INDEX["created_at"]
    return panel

def enforce_time(df):
    assert not df.empty
    assert (df.target_year>df.origin_year).all()
    assert (df.information_cutoff<=df.origin_year).all()
    assert not df.duplicated(["entity_id","origin_year","target_year"]).any()
    assert np.isfinite(df.actual).all()
    return True

def forecast_splits(df):
    years=sorted(df.target_year.dropna().unique().tolist())
    if len(years)<3:return None,"Менее трёх годов с целевыми метками для train/development/test"
    ydev,ytest=years[-2],years[-1]
    tr=df[df.target_year<ydev].copy();dev=df[df.target_year==ydev].copy();test=df[df.target_year==ytest].copy()
    if len(tr)<CFG.MIN_TRAIN_ROWS or len(dev)<CFG.MIN_TEST_ROWS or len(test)<CFG.MIN_TEST_ROWS:
        return None,f"Мало train/dev/test: {len(tr)}/{len(dev)}/{len(test)}"
    assert tr.target_year.max()<dev.target_year.min()<test.target_year.min()
    return {"train":tr,"dev":dev,"test":test,"dev_year":int(ydev),"test_year":int(ytest)},None

def future_mutation_canary(df,keys,target):
    # изменение ещё не известного значения t+1 не должно менять X(t)
    if df.empty:return False
    sorted_df=df.sort_values(keys+["year"]).copy()
    a=make_temporal_panel(sorted_df,keys,target)
    if a.empty:return False
    b=sorted_df.copy(); maxyear=int(b.year.max()); b.loc[b.year==maxyear,target]=numeric(b.loc[b.year==maxyear,target])+123456
    c=make_temporal_panel(b,keys,target)
    join_keys=["entity_id","origin_year","target_year"]
    pair=a.merge(c,on=join_keys,suffixes=("_a","_b"),validate="one_to_one")
    mask=pair.target_year==maxyear
    if not mask.any():return False
    return bool(np.allclose(pair.loc[mask,"prior_value_a"],pair.loc[mask,"prior_value_b"],equal_nan=True))

# Data gate после подготовки 7 независимых предметных задач, без обращения к финальному test.
TASK_DATA={}
if len(od) and od.year.nunique()>=CFG.MIN_YEARS:
    TASK_DATA["ML-5.1"]=make_temporal_panel(od,["origin_region","destination_region","sector"],"flow_count")
else:issue("ML-5.1","Недостаточно наблюдений OD для годового прогноза","origin_region,destination_region,year,flow_count")
if not org_year.empty and org_year.year.nunique()>=CFG.MIN_YEARS:
    TASK_DATA["ML-5.2"]=make_temporal_panel(org_year,["entity_id","sector"],"admissions",features=["group_id"])
else:issue("ML-5.2","Нет утверждённых точных показателей зачисления или недостаточная годовая панель","id, year, admission_count, institution/branch")
if len(od):
    outbound=od.groupby(["origin_region","sector","year"],as_index=False,dropna=False).flow_count.sum()
    outbound=outbound.rename(columns={"flow_count":"outflow_in_observed_pairs"})
    PANELS["outflow_observed"]=outbound
    if outbound.year.nunique()>=CFG.MIN_YEARS:
        TASK_DATA["ML-5.3"]=make_temporal_panel(outbound,["origin_region","sector"],"outflow_in_observed_pairs")
    else:issue("ML-5.3","Недостаточно лет OD; доля от всех абитуриентов недоступна","полный знаменатель для доли выбывших")
if program.empty or not CFG.APPROVED_SPO_ADMISSIONS:
    issue("ML-5.4","Не определено по исходному показателю, какие строки программ являются поступлением","точный indicator_name, crosswalk классификаторов")
else:
    sel=program.loc[program.indicator_name.isin(CFG.APPROVED_SPO_ADMISSIONS)].copy()
    if len(sel) and not sel.duplicated(["region_name","specialty_code","year"]).any() and sel.year.nunique()>=CFG.MIN_YEARS:
        TASK_DATA["ML-5.4"]=make_temporal_panel(sel.rename(columns={"value":"programme_admissions"}),["region_name","specialty_code","sector"],"programme_admissions")
    else:issue("ML-5.4","Нельзя однозначно трактовать программу или размерность исходных строк","programme×region×year, admission_count")
issue("ML-5.5","E04 фиксирует измерение занятости по состоянию на 2024-12-31; ряд независимых дат измерения пока не подтверждён. География работы отсутствует","employment_observation_year, employment_location для retention")
if od.year.nunique()<CFG.NETWORK_MIN_YEARS:issue("ML-5.6","Нет нескольких достоверных годовых сетей","origin,destination,year,flow_count")
# ML-5.7 разрешается только после независимой верификации географии организаций, ниже.
print("Предрегистрация:",PLAN_HASH[:16],"· прогнозные панели:",{k:len(v) for k,v in TASK_DATA.items()})

## 9. Прогнозные модели, пространственные тесты, независимые контрольные группы

Исторический baseline, регуляризованная линейная модель и CatBoost обучаются на **одинаковом** наборе обучающих наблюдений. Выбор CatBoost/Ridge выполняется на development, тестовые ошибки вычисляются после фиксации протокола. Категориальная кодировка обучается только на train. Стохастические повторения выполняются только для CatBoost, а не для детерминированного baseline и Ridge.

Дополнительные тесты `TERRITORY_ONLY`/`CROSSED` выполняются лишь при достаточном числе независимых групп, не смешивая филиалы одного вуза и исходную территорию с целевой.

In [ ]:
def feature_columns(df):
    nums=[c for c in ["prior_value","origin_year","prev_population","prev_migration_rate","number_vo_campuses"] if c in df.columns]
    cats=[c for c in ["origin_region","destination_region","sector","group_id","region","specialty_code"] if c in df.columns]
    return nums,cats

def model_inputs(data,train,seed,kind="ridge"):
    nums,cats=feature_columns(data)
    if not nums:raise ValueError("Нет известных на origin числовых признаков")
    X=data[nums+cats].copy()
    for col in nums:X[col]=numeric(X[col])
    for col in cats:X[col]=X[col].fillna("__MISSING__").astype(str)
    y=data["actual"].to_numpy(dtype=float)
    if kind=="ridge":
        transforms=[("n",Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("scaler",StandardScaler())]),nums)]
        if cats:transforms.append(("c",OneHotEncoder(handle_unknown="ignore"),cats))
        prep=ColumnTransformer(transforms,sparse_threshold=0.4)
        pipe=Pipeline([("prep",prep),("ridge",Ridge(alpha=100.0,solver="lsqr"))])
        tr_x=train[nums+cats].copy()
        for c in cats:tr_x[c]=tr_x[c].fillna("__MISSING__").astype(str)
        pipe.fit(tr_x,train.actual.to_numpy(float))
        if CFG.MODE=="FULL" and int(seed)==int(CFG.FINAL_SEEDS[0]) and ACTIVE_MODULE:
            import joblib
            fp=OUT/"derived"/(ACTIVE_MODULE.replace("-", "_").replace(".", "_")+"_ridge.joblib")
            joblib.dump(pipe,fp,compress=3);MODEL_FILES[str(fp.name)]=fp
        return pipe.predict(X),{"kind":"ridge","alpha":100.0}
    if kind=="catboost":
        train_x=train[nums+cats].copy();x=X.copy()
        for c in nums:
            v=float(pd.to_numeric(train_x[c],errors="coerce").median())
            if not np.isfinite(v):v=0.0
            train_x[c]=pd.to_numeric(train_x[c],errors="coerce").fillna(v)
            x[c]=pd.to_numeric(x[c],errors="coerce").fillna(v)
        for c in cats:
            train_x[c]=train_x[c].fillna("__MISSING__").astype(str);x[c]=x[c].fillna("__MISSING__").astype(str)
        clf=CatBoostRegressor(iterations=90 if CFG.MODE=="SMOKE" else 320,depth=5,learning_rate=0.055,loss_function="RMSE",random_seed=int(seed),verbose=False,thread_count=2,l2_leaf_reg=8.0)
        clf.fit(train_x,train.actual.to_numpy(float),cat_features=cats)
        if CFG.MODE=="FULL" and int(seed)==int(CFG.FINAL_SEEDS[0]) and ACTIVE_MODULE:
            fp=OUT/"derived"/(ACTIVE_MODULE.replace("-", "_").replace(".", "_")+"_catboost.cbm")
            clf.save_model(str(fp));MODEL_FILES[str(fp.name)]=fp
        out=clf.predict(x)
        return out,{"kind":"catboost","seed":int(seed),"iterations":clf.tree_count_,"features":nums+cats,"feature_importance":dict(zip(nums+cats,clf.feature_importances_.tolist()))}
    raise ValueError(kind)

def predict_using(train,test,model,seed=17):
    if model=="baseline":return test.prior_value.to_numpy(dtype=float),{"kind":"persistence"}
    pred,meta=model_inputs(test,train,seed,kind=model)
    return pred,meta

def err(y,yp):
    mask=np.isfinite(y)&np.isfinite(yp)
    y=np.asarray(y)[mask];yp=np.asarray(yp)[mask]
    if len(y)==0:return {"MAE":np.nan,"RMSE":np.nan,"WAPE":np.nan,"n":0}
    den=np.abs(y).sum()
    return {"MAE":float(np.mean(np.abs(y-yp))),"RMSE":float(np.sqrt(np.mean((y-yp)**2))),"WAPE":float(np.abs(y-yp).sum()/den) if den>0 else np.nan,"n":int(len(y))}

def paired_block_ci(df,group_col="bootstrap_group",rep=None,seed=2026):
    rep=rep or (50 if CFG.MODE=="SMOKE" else CFG.BOOTSTRAP_REPS)
    groups=df[group_col].astype(str).unique();rng=np.random.default_rng(seed)
    baseline=np.abs(df.actual-df.pred_baseline).to_numpy(float)
    proposed=np.abs(df.actual-df.pred_model).to_numpy(float)
    bygrp=[np.flatnonzero(df[group_col].astype(str).to_numpy()==g) for g in groups]
    if len(bygrp)<5:return {"delta_MAE":np.nan,"ci_low":np.nan,"ci_high":np.nan,"p_two_sided":np.nan,"groups":len(groups)}
    draws=[]
    for _ in range(rep):
        picks=rng.integers(0,len(groups),len(groups))
        ix=np.concatenate([bygrp[i] for i in picks])
        draws.append(float((proposed[ix]-baseline[ix]).mean()))
    arr=np.asarray(draws)
    return {"delta_MAE":float((proposed-baseline).mean()),"ci_low":float(np.quantile(arr,0.025)),"ci_high":float(np.quantile(arr,0.975)),"p_two_sided":float(min(1.0,2*min((arr<=0).mean(),(arr>=0).mean()))),"groups":len(groups)}

def run_predictive_module(module,frame,cluster_col,nonnegative=True):
    global ACTIVE_MODULE
    ACTIVE_MODULE=module
    if frame.empty:issue(module,"Нет модельной панели");return
    enforce_time(frame)
    packs,why=forecast_splits(frame)
    if packs is None:issue(module,why,"несколько лет train/dev/test");return
    assert future_mutation_canary_for_panel(frame),"Канарейка будущих данных провалена"
    tr,dev,test=packs["train"],packs["dev"],packs["test"]
    # Development (отбор): ни одна оценка test здесь недоступна.
    pdev={};dev_scores={}
    for model in ["baseline","ridge","catboost"]:
        seeds=[CFG.DEV_SEEDS[0]] if model!="catboost" or CFG.MODE=="SMOKE" else list(CFG.DEV_SEEDS)
        scores=[]
        for seed in seeds:
            raw,meta=predict_using(tr,dev,model,seed)
            pred=np.maximum(0,raw) if nonnegative else raw
            scores.append(err(dev.actual.to_numpy(float),pred)["MAE"])
        dev_scores[model]=float(np.mean(scores))
    winner=min(dev_scores,key=dev_scores.get)
    fulltrain=pd.concat([tr,dev],ignore_index=True)
    seed_count=CFG.MAX_FINAL_SEEDS_SMOKE if CFG.MODE=="SMOKE" else (len(CFG.FINAL_SEEDS) if winner=="catboost" else 1)
    predictions={}; allmeta=[]
    for model in ["baseline","ridge","catboost"]:
        seeds=[CFG.FINAL_SEEDS[0]] if model!="catboost" else list(CFG.FINAL_SEEDS[:seed_count])
        for seed in seeds:
            raw,meta=predict_using(fulltrain,test,model,seed)
            pred=np.maximum(0,raw) if nonnegative else raw
            e=err(test.actual.to_numpy(float),pred)
            METRICS.append({"module":module,"protocol":"ROLLING_TIME","model":model,"seed":int(seed) if model=="catboost" else -1,"selected_on_dev":model==winner,"dev_mae_mean":dev_scores[model],"target_year":int(packs["test_year"]),**e})
            out=test[[c for c in test.columns if c in ["entity_id","origin_year","target_year","horizon","information_cutoff","source_vintage","origin_region","destination_region","sector","group_id","region"]]].copy()
            out["module"]=module;out["split_id"]="ROLLING_TIME";out["model"]=model;out["seed"]=int(seed) if model=="catboost" else -1
            out["actual"]=test.actual.to_numpy(float);out["predicted"]=pred
            predictions[(model,int(seed))]=out
            allmeta.append({"module":module,**meta})
    PREDICTIONS.extend(predictions.values());MODELS_META.extend(allmeta)
    # Paired uncertainty using common keys. Stochastic results average over their seeds for primary comparison.
    base=predictions[("baseline",int(CFG.FINAL_SEEDS[0]))]
    for model in ["ridge","catboost"]:
        compared=[x for (m,s),x in predictions.items() if m==model]
        p=np.mean(np.vstack([x.predicted.to_numpy(float) for x in compared]),axis=0)
        comp=pd.DataFrame({"actual":base.actual.to_numpy(float),"pred_baseline":base.predicted.to_numpy(float),"pred_model":p,
                        "bootstrap_group":test[cluster_col].astype(str).to_numpy()})
        ci=paired_block_ci(comp)
        METRICS.append({"module":module,"protocol":"BLOCK_BOOTSTRAP","model":model,"seed":-1,"selected_on_dev":model==winner,"dev_mae_mean":dev_scores[model],"target_year":int(packs["test_year"]),"MAE":err(comp.actual,p)["MAE"],"delta_MAE":ci["delta_MAE"],"ci_low":ci["ci_low"],"ci_high":ci["ci_high"],"p_two_sided":ci["p_two_sided"],"clusters":ci["groups"],"n":len(comp)})
    SPLITS.append({"module":module,"train":len(tr),"dev":len(dev),"test":len(test),"dev_year":packs["dev_year"],"test_year":packs["test_year"],"chosen_dev_model":winner,"group_holdout":"см. отдельную проверку"})
    STATUS[module]={"status":"TESTED_SMOKE" if CFG.MODE=="SMOKE" else "TESTED_FULL","reason":f"Реальный test year {packs['test_year']}; победитель DEV: {winner}; тест вне настройки"}
    return packs

def future_mutation_canary_for_panel(frame):
    # Уже готовая panel: изменение actual в последнем целевом году не должно менять prior_value.
    other=frame.copy();m=other.target_year==other.target_year.max()
    other.loc[m,"actual"]=other.loc[m,"actual"]+999999
    return np.allclose(frame.prior_value.to_numpy(float),other.prior_value.to_numpy(float),equal_nan=True)

# Задача 5.7: E0 и E1 на общей муниципальной поддержке, без искусственного отсутствия вузов.
MUNI_E0_E1=None
if CFG.MODE=="FULL" and not vo_reg.empty and not mun.empty:
    known=vo_reg.dropna(subset=["oktmo","year","entity_id"])
    conf=known.groupby(["oktmo","year"],as_index=False).agg(number_vo_campuses=("entity_id","nunique"),organisations=("entity_id",lambda s:",".join(sorted(set(s.astype(str).tolist()))[:5])))
    # Предыдущий год, а не будущий: E1 не получает информацию об открытии после origin.
    lagged=conf.rename(columns={"year":"edu_year"})
    base=mun[["oktmo","region","year","migration_rate","population"]].dropna(subset=["migration_rate","population"]).copy()
    base=base.rename(columns={"year":"available_year","migration_rate":"prev_migration_rate","population":"prev_population"})
    base["origin_year"]=base.available_year+CFG.PUBLICATION_LAG_YEARS
    base["target_year"]=base.origin_year+1
    later=mun[["oktmo","year","migration_rate"]].rename(columns={"year":"target_year","migration_rate":"actual"})
    base=base.merge(later,on=["oktmo","target_year"],validate="one_to_one")
    base["edu_year"]=base.available_year
    base=base.merge(lagged,on=["oktmo","edu_year"],how="inner",validate="many_to_one")
    base["entity_id"]=base.oktmo.astype(str);base["horizon"]=1
    base["prior_value"]=base.prev_migration_rate
    base["information_cutoff"]=base.available_year
    base["source_vintage"]=INDEX["created_at"]
    ISSUES.append({"module":"ML-5.7","status":"SCIENTIFIC_LIMITATION","reason":"E1 содержит только площадки ВО с точным ОКТМО. Для полного ВО+СПО необходим подтверждённый адрес площадки СПО, совпадающий с ОКТМО муниципалитета","required_fields":"verified SPO municipality campus location"})
    PANELS["municipality_education_context"]=base
    gate("municipality_education_context",base,["entity_id","origin_year","target_year"])
    if base.entity_id.nunique()>=CFG.MIN_MUNI_E1_SUPPORT and base.target_year.nunique()>=3 and base.number_vo_campuses.nunique()>1:
        MUNI_E0_E1=base
    else:issue("ML-5.7",f"Мало точных ОКТМО совпадений или вариации инфраструктуры: n={base.entity_id.nunique()}, years={base.target_year.nunique()}","validated campus OKTMO, stable municipality period, migration_rate")
else:issue("ML-5.7","Требуются идентифицированные площадки ВО/СПО с точным муниципальным ОКТМО","municipal campus location; matching observations")

# Реальные модельные задачи. Прогресс-бар создаётся один раз после планирования.
MODEL_JOBS=[(k,v) for k,v in TASK_DATA.items() if not v.empty and forecast_splits(v)[0] is not None]

for k,v in TASK_DATA.items():
    if k not in [a for a,_ in MODEL_JOBS] and STATUS[k]["status"]=="PENDING":issue(k,forecast_splits(v)[1] or "недостаточная выборка","train/dev/test")
if MUNI_E0_E1 is not None and forecast_splits(MUNI_E0_E1)[0] is None:issue("ML-5.7","Мало лет/наблюдений на общей поддержке E0/E1","train/dev/test")

# Профиль SMOKE исполняет только доступную задачу OD, без научной публикации.
if CFG.MODE=="SMOKE":MODEL_JOBS=[x for x in MODEL_JOBS if x[0]=="ML-5.1"][:1]
BAR=tqdm(total=len(MODEL_JOBS)+7,desc="ML-05 · модель → QA → публикация",unit="этап",leave=True)
for mod,frame in MODEL_JOBS:
    BAR.set_postfix_str(mod)
    cluster="origin_region" if "origin_region" in frame else ("region" if mod=="ML-5.7" else ("group_id" if "group_id" in frame else "entity_id"))
    try:run_predictive_module(mod,frame,cluster,nonnegative=(mod!="ML-5.7"))
    except Exception as e:
        STATUS[mod]={"status":"FAIL","reason":type(e).__name__+": "+str(e)[:180]}
        (OUT/"traceback.log").write_text(traceback.format_exc(),encoding="utf-8")
        if CFG.MODE=="FULL":raise
    BAR.update(1)
print("Фактические модели:",len(METRICS),"строк оценок;",len(PREDICTIONS),"пакетов предсказаний")

## 10. Модуль ML-5.6: структура и устойчивость наблюдаемых образовательных сетей

Направленные рёбра строятся только из сохранившихся фактических OD-строк. Силы, PageRank и сообщества рассчитаны по каждой годовой сети; наличие нулевых/ненаблюдаемых рёбер не подменяется. Различие классификаторов или определений региона требует отдельного сопоставления.

In [ ]:
NETWORK=[];NET_EDGES=[];NET_SETS={}
if not od.empty and od.year.nunique()>=CFG.NETWORK_MIN_YEARS:
    for yr,sub in od.groupby("year"):
        G=nx.DiGraph()
        for _,row in sub.iterrows():
            if row.flow_count>0:G.add_edge(str(row.origin_region),str(row.destination_region),weight=float(row.flow_count))
        if not G.number_of_edges():continue
        ins=dict(G.in_degree(weight="weight"));outs=dict(G.out_degree(weight="weight"))
        pager=nx.pagerank(G,weight="weight",alpha=0.85,max_iter=300)
        comm=list(nx.algorithms.community.louvain_communities(G.to_undirected(),weight="weight",seed=173))
        commap={node:i for i,part in enumerate(comm) for node in part}
        NET_SETS[int(yr)]={frozenset(c) for c in comm}
        total=sum(float(d["weight"]) for _,_,d in G.edges(data=True))
        for node in G.nodes():NETWORK.append({"year":int(yr),"region_code":node,"inflow_observed":ins.get(node,0.),"outflow_observed":outs.get(node,0.),"pagerank":pager.get(node,0.),"community":commap.get(node,-1),"total_observed_flow":total,"edges":G.number_of_edges()})
        for u,v,d in G.edges(data=True):NET_EDGES.append({"year":int(yr),"origin_region":u,"destination_region":v,"flow_count":d["weight"]})
    nodes=pd.DataFrame(NETWORK)
    PANELS["edu_network_year"]=nodes
    PANELS["edu_network_edges"]=pd.DataFrame(NET_EDGES)
    gate("edu_network_year",nodes,["region_code","year"])
    if nodes.year.nunique()>=2:
        years=sorted(NET_SETS)
        jacc=[]
        for a,b in zip(years[:-1],years[1:]):
            p,q=NET_SETS[a],NET_SETS[b]
            common=set.union(*(map(set,p))) & set.union(*(map(set,q))) if p and q else set()
            if common:
                pairs_a={tuple(sorted((u,v))) for part in p for u in part for v in part if u<v and u in common and v in common}
                pairs_b={tuple(sorted((u,v))) for part in q for u in part for v in part if u<v and u in common and v in common}
                denom=len(pairs_a|pairs_b)
                jacc.append({"year_from":a,"year_to":b,"pairwise_community_jaccard":len(pairs_a&pairs_b)/denom if denom else np.nan,"common_regions":len(common)})
        PANELS["network_stability"]=pd.DataFrame(jacc)
        STATUS["ML-5.6"]={"status":"DESCRIPTIVE_PASS","reason":f"Наблюдаемые сети: {len(years)} лет; центральности, сообщества, устойчивость между годами"}
    else:issue("ML-5.6","Недостаточно независимых годов для устойчивости сетей")
else:issue("ML-5.6","Недостаточно реальных OD для наблюдаемой пространственной сети")
BAR.update(1)

## 11. Групповые и пространственные тесты, статистическая неопределённость

На реальных тестовых ключах применяется блочный bootstrap по региону происхождения или по родительской образовательной организации (как задокументировано). Проверяется чувствительность к отбору крупнейших потоков. Семь гипотез не агрегируются в один score. Для заранее заявленных семи независимых сравнений применяется Holm; интерпретация остаётся исследовательской даже при положительной метрике.

In [ ]:
# Group/CROSSED на исходных данных только для достаточных групп; тест независим от временного test.
def group_protocol(module,df,group_col):
    groups=sorted(df[group_col].astype(str).unique())
    if len(groups)<12:return {"module":module,"protocol":"TERRITORY_ONLY","status":"NOT_FEASIBLE","reason":"Мало независимых групп"}
    hashed={g:int(hashlib.sha256((module+g).encode()).hexdigest()[:8],16)%5==0 for g in groups}
    test_mask=df[group_col].astype(str).map(hashed)
    lastyear=int(df.target_year.max())
    train=df[(~test_mask)&(df.target_year<lastyear)].copy()
    test=df[(test_mask)&(df.target_year==lastyear)].copy()
    if len(train)<CFG.MIN_TRAIN_ROWS or len(test)<CFG.MIN_TEST_ROWS:return {"module":module,"protocol":"CROSSED","status":"NOT_FEASIBLE","reason":"Недостаточно отложенных территорий в будущем году"}
    assert set(train[group_col].astype(str)).isdisjoint(set(test[group_col].astype(str)))
    b, _=predict_using(train,test,"baseline")
    m, _=predict_using(train,test,"ridge")
    eb, em=err(test.actual,b),err(test.actual,m)
    return {"module":module,"protocol":"CROSSED","status":"TESTED","train":len(train),"test":len(test),"group_count_train":train[group_col].nunique(),"group_count_test":test[group_col].nunique(),"baseline_MAE":eb["MAE"],"ridge_MAE":em["MAE"],"delta_MAE":em["MAE"]-eb["MAE"]}

GROUP_PROTOCOLS=[]
if CFG.MODE=="FULL":
    for mod,frame in TASK_DATA.items():
        if len(frame):
            col="origin_region" if "origin_region" in frame else "group_id" if "group_id" in frame else None
            if col:GROUP_PROTOCOLS.append(group_protocol(mod,frame,col))
PANELS["group_protocols"]=pd.DataFrame(GROUP_PROTOCOLS)

# H5.7: два одинаковых алгоритма Ridge, различаются только в доступности образовательного признака.
def run_e0_e1(base):
    pack,why=forecast_splits(base)
    if pack is None:issue("ML-5.7",why);return
    train=pd.concat([pack["train"],pack["dev"]],ignore_index=True)
    test=pack["test"].copy()
    covars0=["prior_value","prev_population","origin_year"]
    covars1=covars0+["number_vo_campuses"]
    estimators={}
    for name,cols in [("E0",covars0),("E1",covars1)]:
        x=train[cols].apply(pd.to_numeric,errors="coerce")
        xt=test[cols].apply(pd.to_numeric,errors="coerce")
        model=Pipeline([("impute",SimpleImputer(strategy="median",add_indicator=True)),("scale",StandardScaler()),("ridge",Ridge(alpha=100.0))])
        model.fit(x,train.actual.astype(float))
        yp=model.predict(xt);estimators[name]=yp
        e=err(test.actual,yp)
        METRICS.append({"module":"ML-5.7","protocol":"ROLLING_TIME_COMMON_SUPPORT","model":name,"seed":-1,"target_year":int(pack["test_year"]),**e})
        p=test[["entity_id","origin_year","target_year","horizon","information_cutoff","source_vintage","region"]].copy()
        p["actual"]=test.actual.astype(float);p["predicted"]=yp;p["module"]="ML-5.7";p["model"]=name;p["seed"]=-1;p["split_id"]="E0_E1_common_support"
        PREDICTIONS.append(p)
    comp=pd.DataFrame({"actual":test.actual.to_numpy(float),"pred_baseline":estimators["E0"],"pred_model":estimators["E1"],"bootstrap_group":test.region.astype(str)})
    delta=paired_block_ci(comp)
    METRICS.append({"module":"ML-5.7","protocol":"BLOCK_BOOTSTRAP","model":"E1_minus_E0","seed":-1,"target_year":int(pack["test_year"]),"n":len(test),**delta})
    SPLITS.append({"module":"ML-5.7","train":len(pack["train"]),"dev":len(pack["dev"]),"test":len(test),"dev_year":pack["dev_year"],"test_year":pack["test_year"],"chosen_dev_model":"not_selected; fixed E0/E1"})
    STATUS["ML-5.7"]={"status":"TESTED_FULL","reason":"E0/E1 проверены на точном общем множестве ОКТМО и годовых меток; отсутствие СПО не приписывается"}

if CFG.MODE=="FULL" and MUNI_E0_E1 is not None:run_e0_e1(MUNI_E0_E1)

# Препроцессинг задним числом не гарантирует доступность на дату прогноза: фиксируем ограничение.
ISSUES.append({"module":"ALL","status":"SCIENTIFIC_LIMITATION","reason":"Нет пообъектных available_at: историческая валидация при консервативном publication lag; не выдавать за доказанную реалистичную vintage-репликацию","required_fields":"publication_year / available_at per field"})

metrics_df=pd.DataFrame(METRICS)
if not metrics_df.empty and "p_two_sided" in metrics_df:
    confirm=metrics_df.loc[(metrics_df.protocol=="BLOCK_BOOTSTRAP") & metrics_df.p_two_sided.notna()].copy()
    confirm["holm_p"]=np.nan
    if len(confirm):
        p=np.asarray(confirm.p_two_sided,dtype=float);idx=np.argsort(p);adj=np.empty_like(p)
        for rank,j in enumerate(idx):adj[j]=min(1.0,p[j]*(len(p)-rank))
        adj[idx]=np.maximum.accumulate(adj[idx]);confirm["holm_p"]=adj
        metrics_df=metrics_df.merge(confirm[["module","model","protocol","holm_p"]],on=["module","model","protocol"],how="left",validate="many_to_one")
PANELS["metrics"]=metrics_df
print("Статусы по семи модулям:",{k:v["status"] for k,v in STATUS.items()})
BAR.update(1)

## 12. Визуализация реальных данных и интерпретация

Рисунки имеют единицы измерения, период и принадлежность первоисточнику. При отсутствии подтверждённого показателя не создаётся декоративная «карта» или пустая диаграмма. Отклонение прогноза указывает на статистическую ошибку, а не на эффект вмешательства в управление приёмом.

In [ ]:
def plot_save(name):
    fp=OUT/"figures"/(name+".png")
    plt.tight_layout();plt.savefig(fp,dpi=150,bbox_inches="tight");plt.close()
    FIGURES.append(str(fp));return fp

if not od.empty:
    yr=int(od.year.max());e=od.loc[od.year==yr].copy()
    top=e.groupby("destination_region",as_index=False).flow_count.sum().nlargest(15,"flow_count")
    plt.figure(figsize=(10,5));plt.bar(top.destination_region.astype(str),top.flow_count)
    plt.xticks(rotation=60,ha="right");plt.ylabel("Наблюдаемый поток, чел. (как в E01)")
    plt.title(f"СПО: крупнейшие регионы назначения, {yr} · зарегистрированные OD");plot_save("01_spo_destination_top15")
    origins=e.groupby("origin_region").flow_count.sum().nlargest(12).index
    dests=e.groupby("destination_region").flow_count.sum().nlargest(12).index
    heat=e.loc[e.origin_region.isin(origins)&e.destination_region.isin(dests)].pivot_table(index="origin_region",columns="destination_region",values="flow_count",aggfunc="sum")
    plt.figure(figsize=(10,7));plt.imshow(heat.fillna(0).to_numpy(),aspect="auto")
    plt.xticks(range(len(heat.columns)),heat.columns,rotation=90,fontsize=7);plt.yticks(range(len(heat.index)),heat.index,fontsize=7)
    plt.colorbar(label="Наблюдаемые OD-значения; пустые клетки визуально 0 только для цвета")
    plt.title(f"СПО: подматрица крупнейших зарегистрированных OD, {yr}");plot_save("02_spo_od_matrix_observed_only")
if PREDICTIONS:
    pred=pd.concat(PREDICTIONS,ignore_index=True)
    p=pred.loc[(pred.module=="ML-5.1")&(pred.model=="ridge")]
    if len(p):
        p=p.nlargest(min(400,len(p)),"actual")
        plt.figure(figsize=(7,6));plt.scatter(p.actual,p.predicted,s=8,alpha=0.5)
        lim=max(float(p.actual.max()),float(p.predicted.max()),1)
        plt.plot([0,lim],[0,lim],linestyle="--",linewidth=1)
        plt.xlabel("Факт, наблюдаемый OD");plt.ylabel("Прогноз Ridge")
        plt.title("СПО: крупнейшие OD на финальном временном тесте");plot_save("03_spo_observed_vs_predicted")
    p=pred.loc[(pred.module=="ML-5.7")]
    if len(p):
        sums=p.assign(abs_error=(p.actual-p.predicted).abs()).groupby("model").abs_error.mean()
        plt.figure(figsize=(5,4));plt.bar(sums.index,sums.values)
        plt.ylabel("MAE, миграционный коэффициент муниципалитета");plt.title("E0 против E1: общая тестовая поддержка")
        plot_save("04_municipal_e0_e1")
BAR.update(1)

# Интерпретация строится только по фактическим сравнениям.
INTERPRETATION=[]
for mod in STATUS:
    sub=metrics_df.loc[metrics_df.module==mod] if not metrics_df.empty else pd.DataFrame()
    if sub.empty:
        INTERPRETATION.append({"module":mod,"conclusion":"Независимое сравнение моделей отсутствует; " + STATUS[mod]["reason"],"evidence":"DATA_GATE/статус"})
        continue
    win=sub.loc[sub.protocol=="ROLLING_TIME"]
    if len(win):
        b=win.loc[win.model=="baseline"]
        if not b.empty:
            baseline_mae=float(b.iloc[0].MAE)
            best=win.loc[win.MAE==win.MAE.min()].iloc[0]
            msg=f"На независимом временном test лучший по MAE: {best.model} ({best.MAE:.4g}) против инерции ({baseline_mae:.4g}). Сведения об опубликованности признаков на дату прогноза неполны."
        else:msg="Оценка выполнена, но исторический baseline отсутствует"
    else:msg="Проверены E0/E1 на общей выборке без утверждения причинного эффекта"
    INTERPRETATION.append({"module":mod,"conclusion":msg,"evidence":"метрики + прогнозы в evidence"})
PANELS["interpretation"]=pd.DataFrame(INTERPRETATION)
BAR.update(1)

## 13. Экспорт: Excel, технический ZIP и доказательные таблицы

Формируются ровно четыре логических результата: сам ноутбук, один Excel, один ZIP и `run_manifest.json`. Полные панели и построчные прогнозы сохраняются в Parquet внутри ZIP. Excel содержит реальные компактные таблицы/метрики; недоступные исследования перечисляются на листе ограничений, без пустых заготовок.

In [ ]:
import zipfile, tempfile
EXCEL=OUT/"ML5_EDUCATION_MIGRATION_RESULTS.xlsx"
EVIDENCE=OUT/"ML5_EVIDENCE.zip"
MANIFEST=OUT/"run_manifest.json"
NOTEBOOK=OUT/"ML5_EDUCATION_MIGRATION.ipynb"

def sheet_frame(df,book,title,max_rows=20000):
    if not isinstance(df,pd.DataFrame) or df.empty:return
    name=title[:31]
    safe=df.head(max_rows).copy()
    for c in safe.columns:
        if pd.api.types.is_datetime64_any_dtype(safe[c]):safe[c]=safe[c].astype(str)
        if pd.api.types.is_object_dtype(safe[c]):safe[c]=safe[c].map(lambda x:json.dumps(x,ensure_ascii=False,default=str) if isinstance(x,(dict,list,tuple,set)) else str(x) if x is not None else "")
    safe.to_excel(book,sheet_name=name,index=False)
    ws=book.sheets[name];ws.freeze_panes(1,0)
    last=max(0,len(safe));ws.autofilter(0,0,last,len(safe.columns)-1)
    fmt=book.book.add_format({"bold":True,"font_color":"#FFFFFF","bg_color":"#253C58","text_wrap":True,"valign":"vcenter","border":0})
    ws.set_row(0,30,fmt)
    for i,c in enumerate(safe.columns):
        sample=safe[c].astype(str).head(200)
        width=min(41,max(12,len(str(c))+2,int(sample.str.len().quantile(0.85))+2 if len(sample) else 12))
        ws.set_column(i,i,width)
    ws.set_tab_color("#2A6F8D")

rows_status=[{"module":k,**v,"hypothesis":"H"+k.split("-")[1]} for k,v in STATUS.items()]
passport=pd.DataFrame([
    {"Параметр":"Исследование","Значение":"ML-05 Образовательная миграция"},
    {"Параметр":"Режим","Значение":CFG.MODE},
    {"Параметр":"Версия","Значение":CFG.VERSION},
    {"Параметр":"Приватный Stage I","Значение":CFG.REPO},
    {"Параметр":"Каталог","Значение":CFG.INDEX},
    {"Параметр":"Источники","Значение":str(len(ASSETS))},
    {"Параметр":"Подтверждённые СПО OD","Значение":str(len(od))},
    {"Параметр":"Ограничение OD","Значение":"Место получения предыдущего образования/жительства по E01 требует первичного определения; нет переноса в постоянную миграцию"},
    {"Параметр":"Защита информации","Значение":"Ненаблюдаемые связи, география трудоустройства и нули не реконструировались"},
    {"Параметр":"Состояние публикации","Значение":"SMOKE — запрещена; FULL — только после локального QA"},
])
issues_df=pd.DataFrame(ISSUES)
with pd.ExcelWriter(EXCEL,engine="xlsxwriter",engine_kwargs={"options":{"strings_to_formulas":False,"strings_to_urls":False}}) as writer:
    sheet_frame(passport,writer,"Паспорт")
    sheet_frame(pd.DataFrame(rows_status),writer,"Семь_модулей")
    sheet_frame(ASSET_AUDIT,writer,"Источники_ВО_СПО")
    sheet_frame(pd.DataFrame(QA),writer,"DATA_GATE")
    sheet_frame(pd.DataFrame(PROVENANCE),writer,"Происхождение_данных")
    sheet_frame(issues_df,writer,"Пределы_данных")
    sheet_frame(pd.DataFrame(INTERPRETATION),writer,"Семь_гипотез")
    sheet_frame(metrics_df,writer,"Метрики_и_CI")
    sheet_frame(pd.DataFrame(SPLITS),writer,"Разбиения")
    sheet_frame(PANELS.get("group_protocols"),writer,"Пространственный_тест")
    sheet_frame(PANELS.get("edu_org_registry"),writer,"Организации_филиалы",max_rows=4000)
    sheet_frame(PANELS.get("edu_org_year"),writer,"ВО_СПО_прием",max_rows=10000)
    sheet_frame(PANELS.get("edu_od_year"),writer,"OD_потоки_СПО",max_rows=10000)
    sheet_frame(PANELS.get("outflow_observed"),writer,"Отток_регионов",max_rows=10000)
    sheet_frame(PANELS.get("edu_program_year"),writer,"Программы",max_rows=10000)
    sheet_frame(PANELS.get("graduate_outcomes"),writer,"Выпускники_занятость",max_rows=10000)
    sheet_frame(PANELS.get("edu_network_year"),writer,"Сеть_регионы",max_rows=20000)
    sheet_frame(PANELS.get("network_stability"),writer,"Сеть_по_годам")
    sheet_frame(PANELS.get("municipality_education_context"),writer,"Муниципалитеты_E0_E1",max_rows=10000)
    sheet_frame(PANELS.get("spo_indicator_dictionary"),writer,"Словарь_СПО",max_rows=2000)
    sheet_frame(PANELS.get("e02_indicator_dictionary"),writer,"Словарь_ВО",max_rows=2000)

METHODS="""# ML-05. Методы, единицы наблюдения и ограничения

Происхождение данных: handoff/07_v2/dataset_index.json; все SHA записаны в sources.json.
OD: СПО; матрица может требовать дополнительных определений origin. Не реконструировать отсутствующие ребра.
Для количественных целей MAE и RMSE. WAPE агрегируется с защитой делителя от нуля.
Временная валидация: train до development; test последний год, не участвует в выборе гиперпараметров.
Baseline: прошлый наблюдаемый показатель; Ridge: регуляризованная линейная регрессия; CatBoost: бустинг.
Категориальная кодировка и числовая импутация обучаются только на train.
Интервалы: кластерный bootstrap по регионам или группам организаций. Holm — подтверждающие сравнения.
E0/E1: одинаковая Ridge, одинаковые муниципалитеты/годы; в E1 дополнительно число подтверждённых площадок ВО.
Миграция ради образования не равна изменению постоянного места жительства.
Рост прогностической точности не идентифицирует причинный эффект открытия колледжа или вуза.
Неподтверждённые годы публикации позволяют только ретроспективную проверку при предположении о лагах.
"""

EVIDENCE.unlink(missing_ok=True)
PRED=pd.concat(PREDICTIONS,ignore_index=True) if PREDICTIONS else pd.DataFrame()
with zipfile.ZipFile(EVIDENCE,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    z.writestr("research_methods.md",METHODS)
    z.writestr("metadata/config.json",json.dumps(asdict(CFG),ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/experiment_plan.json",json.dumps(EXPERIMENT_PLAN,ensure_ascii=False,indent=2))
    z.writestr("metadata/provenance.json",json.dumps(PROVENANCE,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/qa.json",json.dumps(QA,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/modules.json",json.dumps(STATUS,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/limitations.json",json.dumps(ISSUES,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/model_parameters.json",json.dumps(MODELS_META,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/splits.json",json.dumps(SPLITS,ensure_ascii=False,default=str,indent=2))
    for name,data in PANELS.items():
        if isinstance(data,pd.DataFrame) and not data.empty:
            path=OUT/"derived"/(name+".parquet")
            try:data.to_parquet(path,index=False)
            except (pa.ArrowInvalid,TypeError,ValueError) as ex:
                raise RuntimeError("Не удалось сериализовать доказательную панель "+name+": "+str(ex)[:120])
            z.write(path,"panels/"+path.name)
    if not PRED.empty:
        predpath=OUT/"derived"/"education_predictions.parquet"
        PRED.to_parquet(predpath,index=False);z.write(predpath,"predictions/education_predictions.parquet")
    for path in FIGURES:z.write(path,"figures/"+Path(path).name)
    for name,path in MODEL_FILES.items():z.write(path,"models/"+name)
    if (OUT/"traceback.log").exists():z.write(OUT/"traceback.log","run.log")
    else:z.writestr("run.log","No unhandled exceptions recorded in this execution.\n")
BAR.update(1)
print("Созданы локальные результаты:",EXCEL.name,round(EXCEL.stat().st_size/1e6,2),"MB;",EVIDENCE.name,round(EVIDENCE.stat().st_size/1e6,2),"MB")

## 14–15. Контроль целостности и публикация в отдельной ветке

Сам ноутбук восстанавливается **из встроенного состава ячеек**, без Google Drive, внешнего `.py` и ручного копирования. Финальная выгрузка проходит `nbformat.validate`, сканирование секретов, проверку Excel, ZIP и каждого Parquet. `SMOKE` неизменно запрещает итоговую публикацию. При `FULL QA` данные публикуются **только в новом сегменте отдельной ветки**, тяжёлый evidence — в Release того же приватного репозитория; опубликованные байты скачиваются повторно и сверяются с локальными SHA-256. Если публикация не подтверждена — `PUBLICATION_FAIL`, а не фиктивный `PASS_GITHUB`.

In [ ]:
# Самодостаточная точная реконструкция исходного .ipynb без Colab/Drive API.
_BUNDLE_B64 = '...'
_SELF_TEMPLATE = '# Самодостаточная точная реконструкция исходного .ipynb без Colab/Drive API.\n_BUNDLE_B64 = %r\n_SELF_TEMPLATE = %r\nimport io, zlib\n_prefix=json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)))\n_this_cell=(_SELF_TEMPLATE %% (_BUNDLE_B64,_SELF_TEMPLATE)).strip()\n_clean=nbformat.v4.new_notebook(cells=[nbformat.from_dict(x) for x in _prefix]+[nbformat.v4.new_code_cell(_this_cell)],metadata={"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python"},"colab":{"provenance":[]}})\nfor cell in _clean.cells:\n    if cell.cell_type=="code":\n        cell.outputs=[];cell.execution_count=None\nnbformat.validate(_clean)\nnbformat.write(_clean,str(NOTEBOOK))\nnb_raw=NOTEBOOK.read_text(encoding="utf-8")\nif TOKEN in nb_raw:raise RuntimeError("SECURITY_FAIL: секрет обнаружен в реконструированном файле")\nif re.search(r"(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}",nb_raw):raise RuntimeError("SECURITY_FAIL: обнаружен вид токена")\n\n# Проверка научных выходов, структуры Excel и каждой реальной Parquet-панели.\nwith zipfile.ZipFile(EXCEL) as xlsx:\n    if xlsx.testzip() is not None:raise RuntimeError("Excel ZIP damaged")\n    sheet_files=[n for n in xlsx.namelist() if re.fullmatch(r"xl/worksheets/sheet\\d+\\.xml",n)]\n    assert len(sheet_files)>=4, "Недостаточно заполненных Excel-листов"\n    for item in sheet_files:\n        xml=xlsx.read(item)\n        assert b"<row" in xml, "Пустой лист Excel: "+item\nzip_stats={}\nwith zipfile.ZipFile(EVIDENCE) as z:\n    bad=z.testzip()\n    if bad is not None:raise RuntimeError("Evidence ZIP corrupted: "+bad)\n    parquet_files=[x for x in z.namelist() if x.endswith(".parquet")]\n    assert parquet_files, "Нет сохранённых Parquet-панелей"\n    for member in parquet_files:\n        meta=pq.ParquetFile(io.BytesIO(z.read(member))).metadata\n        if meta.num_rows<1:raise RuntimeError("Пустой Parquet: "+member)\n        zip_stats[member]={"rows":int(meta.num_rows),"columns":int(meta.num_columns)}\n    assert "panels/edu_od_year.parquet" in zip_stats, "Нет исходной подготовленной OD-панели"\n    if PREDICTIONS:assert "predictions/education_predictions.parquet" in zip_stats\nif len(PRED):\n    key=[c for c in ["module","entity_id","origin_year","target_year","split_id","model","seed"] if c in PRED.columns]\n    assert not PRED.duplicated(key).any(), "Дубли модельных прогнозов"\n    assert (PRED.target_year>PRED.origin_year).all(), "Нарушен cutoff"\nif CFG.MODE=="FULL":\n    eligible=[k for k in STATUS if STATUS[k]["status"] in ["TESTED_FULL","DESCRIPTIVE_PASS"]]\n    if not eligible:raise RuntimeError("FULL_BLOCKED: ни один модуль не завершил научную оценку")\n    not_scientific=[k for k,v in STATUS.items() if v["status"] not in ["TESTED_FULL","DESCRIPTIVE_PASS","NOT_FEASIBLE"]]\n    if not_scientific:raise RuntimeError("FULL_BLOCKED: неразрешённые статусы "+str(not_scientific))\n\n# Реально полученные локальные хеши — не подменять индексную верификацию подтверждением всего ML-5.\nINPUT_HASHES={}\nfor a in ASSETS:\n    p=OUT/"inputs"/(hashlib.sha256(a["asset_id"].encode()).hexdigest()[:20]+".parquet")\n    if p.exists():\n        if sha256(p)!=a["sha256"]:raise RuntimeError("Повреждён кэш исходника: "+a["asset_id"])\n        INPUT_HASHES[a["asset_id"]]=a["sha256"]\nOUTPUT_HASHES={f.name:sha256(f) for f in [EXCEL,EVIDENCE,NOTEBOOK]}\nFULL_QA=(CFG.MODE=="FULL") and bool(METRICS) and (len(zip_stats)>=2)\nRUN_STATUS="FULL_LOCAL_QA_PASS" if FULL_QA else "SMOKE_ONLY" if CFG.MODE=="SMOKE" else "PARTIAL"\nRUN_MANIFEST={"package_versions":{p:__import__("importlib.metadata",fromlist=["version"]).version(p) for p in ["numpy","pandas","scikit-learn","catboost","pyarrow","duckdb","networkx","XlsxWriter","nbformat"]},"version":CFG.VERSION,"run_status":RUN_STATUS,"mode":CFG.MODE,"repo":CFG.REPO,"segment":CFG.SEGMENT,"stage1_index_version":INDEX["version"],"index_content_canonical_sha256":hashlib.sha256(json.dumps(INDEX,ensure_ascii=False,sort_keys=True).encode()).hexdigest(),"plan_hash":PLAN_HASH,"input_sha256":INPUT_HASHES,"outputs_sha256":OUTPUT_HASHES,"parquet_zip":zip_stats,"module_status":STATUS,"qa":QA,"models_metrics_count":len(METRICS),"real_predictions_rows":len(PRED),"start_observation_year":int(od.year.min()) if len(od) else None,"end_observation_year":int(od.year.max()) if len(od) else None,"observed_od_count":len(od),"wall_seconds":round(time.monotonic()-RUN_STARTED,2),"git_publication":"NOT_ATTEMPTED","generated_utc":datetime.now(timezone.utc).isoformat()}\nMANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,default=str,indent=2),encoding="utf-8")\nBAR.update(1)\n\ndef github_json(method,url,**kwargs):\n    response=safe_request(method,url,**kwargs)\n    try:return response.json()\n    finally:response.close()\n\ndef github_binary(url):\n    response=safe_request("GET",url,headers={"Accept":"application/octet-stream"},stream=True)\n    if "json" in response.headers.get("content-type","").lower():\n        detail=response.json();response.close()\n        response=safe_request("GET",detail["browser_download_url"],stream=True)\n    h=hashlib.sha256()\n    for part in response.iter_content(2*1024*1024):\n        if part:h.update(part)\n    response.close()\n    return h.hexdigest()\n\ndef github_put_file(branch,path,file):\n    # Файл в новом собственном сегменте. Не редактирует main.\n    payload={"message":"ML-05 verified full research output: "+file.name,"content":base64.b64encode(file.read_bytes()).decode("ascii"),"branch":branch}\n    obj=github_json("PUT",API+"/contents/"+quote(path,safe="/"),json=payload,timeout=(30,900))\n    return obj["commit"]["sha"]\n\ndef github_contents_sha(branch,path):\n    target=API+"/contents/"+quote(path,safe="/")\n    r=safe_request("GET",target,params={"ref":branch})\n    v=r.json();r.close()\n    if v.get("content"):\n        return hashlib.sha256(base64.b64decode(v["content"])).hexdigest()\n    r=safe_request("GET",target,params={"ref":branch},headers={"Accept":"application/vnd.github.raw+json"},stream=True)\n    h=hashlib.sha256()\n    for chunk in r.iter_content(2*1024*1024):\n        if chunk:h.update(chunk)\n    r.close();return h.hexdigest()\n\ndef publish_full():\n    if not FULL_QA or CFG.MODE!="FULL":raise RuntimeError("PUBLICATION_BLOCKED: необходим реальный FULL LOCAL QA")\n    source=github_json("GET",API+"/git/ref/heads/"+quote(CFG.INPUT_REF))\n    sha=source["object"]["sha"]\n    suffix=datetime.now(timezone.utc).strftime("%%Y%%m%%dT%%H%%M%%SZ")\n    branch=CFG.TARGET_BRANCH_PREFIX+"-"+suffix\n    # Отдельная новая ветка, никогда force-push.\n    github_json("POST",API+"/git/refs",json={"ref":"refs/heads/"+branch,"sha":sha})\n    root=CFG.SEGMENT.strip("/")+"/"\n    commits={}\n    for path in [NOTEBOOK,EXCEL]:\n        commits[path.name]=github_put_file(branch,root+path.name,path)\n    artifact_sha=commits[EXCEL.name]\n    # Новый release tag привязан к изолированной ветке. Не перезаписываем существующую версию.\n    release_tag=CFG.VERSION+"-"+suffix\n    rel=github_json("POST",API+"/releases",json={"tag_name":release_tag,"target_commitish":branch,"name":"ML-05 · "+release_tag,"draft":False,"prerelease":True,"generate_release_notes":False,"body":"ML-05: validation-gated evidence for private research; SHA-256 in run_manifest.json"})\n    data=EVIDENCE.read_bytes()\n    upload_url="https://uploads.github.com/repos/"+CFG.REPO+"/releases/"+str(rel["id"])+"/assets"\n    asset=github_json("POST",upload_url,params={"name":EVIDENCE.name},data=data,headers={"Content-Type":"application/zip"},timeout=(30,1800))\n    link=asset["browser_download_url"]\n    if github_binary(API+"/releases/assets/"+str(asset["id"]))!=sha256(EVIDENCE):raise RuntimeError("Проверка Release SHA-256 не пройдена")\n    for file in [NOTEBOOK,EXCEL]:\n        remote=github_contents_sha(branch,root+file.name)\n        if remote!=sha256(file):raise RuntimeError("Опубликованный файл отличается: "+file.name)\n    RUN_MANIFEST["git_publication"]="PASS_GITHUB"\n    RUN_MANIFEST["publication_branch"]=branch\n    RUN_MANIFEST["artifact_commit_sha"]=artifact_sha\n    RUN_MANIFEST["release_tag"]=release_tag\n    RUN_MANIFEST["evidence_release_asset_url"]=link\n    RUN_MANIFEST["notebook_url"]="https://github.com/"+CFG.REPO+"/blob/"+branch+"/"+root+NOTEBOOK.name\n    RUN_MANIFEST["excel_url"]="https://github.com/"+CFG.REPO+"/blob/"+branch+"/"+root+EXCEL.name\n    RUN_MANIFEST["run_status"]="FULL_PASS"\n    MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,default=str,indent=2),encoding="utf-8")\n    commits[MANIFEST.name]=github_put_file(branch,root+MANIFEST.name,MANIFEST)\n    if github_contents_sha(branch,root+MANIFEST.name)!=sha256(MANIFEST):raise RuntimeError("Опубликованный манифест не совпадает")\n    return {"branch":branch,"tag":release_tag,"artifact_commit":artifact_sha,"manifest_commit":commits[MANIFEST.name]}\n\npublished=None\nif CFG.MODE=="FULL" and FULL_QA and CFG.PUBLISH_AFTER_FULL_QA:\n    try: published=publish_full()\n    except Exception as error:\n        RUN_MANIFEST["git_publication"]="PUBLICATION_FAIL"\n        RUN_MANIFEST["run_status"]="PUBLICATION_FAIL"\n        RUN_MANIFEST["publish_error_type"]=type(error).__name__\n        MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,default=str,indent=2),encoding="utf-8")\n        print("PUBLICATION_FAIL: публикация не подтверждена; детали в traceback.log")\n        (OUT/"traceback.log").write_text(traceback.format_exc(),encoding="utf-8")\nelse:\n    print("Публикация пропущена: SMOKE, PARTIAL или явно отключена в конфигурации")\nBAR.update(1)\nBAR.close()\n\n# 15. Краткая сводка фактических проверок.\nprint("\\nML-05 · "+RUN_MANIFEST["run_status"])\nprint("Данные:",len(od),"наблюдаемых строк СПО OD;",len(INPUT_HASHES),"проверенных скачанных исходников")\nprint("Модели:",len(METRICS),"записей оценки;",len(PRED),"тестовых прогнозов;",len(FIGURES),"реальных графиков")\nfor mod,s in STATUS.items():print(mod,s["status"],"—",s["reason"][:130])\nif not metrics_df.empty:\n    for _,r in metrics_df.loc[metrics_df.protocol=="ROLLING_TIME"].head(5).iterrows():print(r.module,r.model,"MAE=",round(float(r.MAE),4))\nif published:print("Публикация:",published)\nprint("Локальные файлы:",NOTEBOOK,EXCEL,EVIDENCE,MANIFEST,sep="\\n")\n'
import io, zlib
_prefix=json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)))
_this_cell=(_SELF_TEMPLATE % (_BUNDLE_B64,_SELF_TEMPLATE)).strip()
_clean=nbformat.v4.new_notebook(cells=[nbformat.from_dict(x) for x in _prefix]+[nbformat.v4.new_code_cell(_this_cell)],metadata={"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python"},"colab":{"provenance":[]}})
for cell in _clean.cells:
    if cell.cell_type=="code":
        cell.outputs=[];cell.execution_count=None
nbformat.validate(_clean)
nbformat.write(_clean,str(NOTEBOOK))
nb_raw=NOTEBOOK.read_text(encoding="utf-8")
if TOKEN in nb_raw:raise RuntimeError("SECURITY_FAIL: секрет обнаружен в реконструированном файле")
if re.search(r"(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}",nb_raw):raise RuntimeError("SECURITY_FAIL: обнаружен вид токена")

# Проверка научных выходов, структуры Excel и каждой реальной Parquet-панели.
with zipfile.ZipFile(EXCEL) as xlsx:
    if xlsx.testzip() is not None:raise RuntimeError("Excel ZIP damaged")
    sheet_files=[n for n in xlsx.namelist() if re.fullmatch(r"xl/worksheets/sheet\d+\.xml",n)]
    assert len(sheet_files)>=4, "Недостаточно заполненных Excel-листов"
    for item in sheet_files:
        xml=xlsx.read(item)
        assert b"<row" in xml, "Пустой лист Excel: "+item
zip_stats={}
with zipfile.ZipFile(EVIDENCE) as z:
    bad=z.testzip()
    if bad is not None:raise RuntimeError("Evidence ZIP corrupted: "+bad)
    parquet_files=[x for x in z.namelist() if x.endswith(".parquet")]
    assert parquet_files, "Нет сохранённых Parquet-панелей"
    for member in parquet_files:
        meta=pq.ParquetFile(io.BytesIO(z.read(member))).metadata
        if meta.num_rows<1:raise RuntimeError("Пустой Parquet: "+member)
        zip_stats[member]={"rows":int(meta.num_rows),"columns":int(meta.num_columns)}
    assert "panels/edu_od_year.parquet" in zip_stats, "Нет исходной подготовленной OD-панели"
    if PREDICTIONS:assert "predictions/education_predictions.parquet" in zip_stats
if len(PRED):
    key=[c for c in ["module","entity_id","origin_year","target_year","split_id","model","seed"] if c in PRED.columns]
    assert not PRED.duplicated(key).any(), "Дубли модельных прогнозов"
    assert (PRED.target_year>PRED.origin_year).all(), "Нарушен cutoff"
if CFG.MODE=="FULL":
    eligible=[k for k in STATUS if STATUS[k]["status"] in ["TESTED_FULL","DESCRIPTIVE_PASS"]]
    if not eligible:raise RuntimeError("FULL_BLOCKED: ни один модуль не завершил научную оценку")
    not_scientific=[k for k,v in STATUS.items() if v["status"] not in ["TESTED_FULL","DESCRIPTIVE_PASS","NOT_FEASIBLE"]]
    if not_scientific:raise RuntimeError("FULL_BLOCKED: неразрешённые статусы "+str(not_scientific))

# Реально полученные локальные хеши — не подменять индексную верификацию подтверждением всего ML-5.
INPUT_HASHES={}
for a in ASSETS:
    p=OUT/"inputs"/(hashlib.sha256(a["asset_id"].encode()).hexdigest()[:20]+".parquet")
    if p.exists():
        if sha256(p)!=a["sha256"]:raise RuntimeError("Повреждён кэш исходника: "+a["asset_id"])
        INPUT_HASHES[a["asset_id"]]=a["sha256"]
OUTPUT_HASHES={f.name:sha256(f) for f in [EXCEL,EVIDENCE,NOTEBOOK]}
FULL_QA=(CFG.MODE=="FULL") and bool(METRICS) and (len(zip_stats)>=2)
RUN_STATUS="FULL_LOCAL_QA_PASS" if FULL_QA else "SMOKE_ONLY" if CFG.MODE=="SMOKE" else "PARTIAL"
RUN_MANIFEST={"package_versions":{p:__import__("importlib.metadata",fromlist=["version"]).version(p) for p in ["numpy","pandas","scikit-learn","catboost","pyarrow","duckdb","networkx","XlsxWriter","nbformat"]},"version":CFG.VERSION,"run_status":RUN_STATUS,"mode":CFG.MODE,"repo":CFG.REPO,"segment":CFG.SEGMENT,"stage1_index_version":INDEX["version"],"index_content_canonical_sha256":hashlib.sha256(json.dumps(INDEX,ensure_ascii=False,sort_keys=True).encode()).hexdigest(),"plan_hash":PLAN_HASH,"input_sha256":INPUT_HASHES,"outputs_sha256":OUTPUT_HASHES,"parquet_zip":zip_stats,"module_status":STATUS,"qa":QA,"models_metrics_count":len(METRICS),"real_predictions_rows":len(PRED),"start_observation_year":int(od.year.min()) if len(od) else None,"end_observation_year":int(od.year.max()) if len(od) else None,"observed_od_count":len(od),"wall_seconds":round(time.monotonic()-RUN_STARTED,2),"git_publication":"NOT_ATTEMPTED","generated_utc":datetime.now(timezone.utc).isoformat()}
MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,default=str,indent=2),encoding="utf-8")
BAR.update(1)

def github_json(method,url,**kwargs):
    response=safe_request(method,url,**kwargs)
    try:return response.json()
    finally:response.close()

def github_binary(url):
    response=safe_request("GET",url,headers={"Accept":"application/octet-stream"},stream=True)
    if "json" in response.headers.get("content-type","").lower():
        detail=response.json();response.close()
        response=safe_request("GET",detail["browser_download_url"],stream=True)
    h=hashlib.sha256()
    for part in response.iter_content(2*1024*1024):
        if part:h.update(part)
    response.close()
    return h.hexdigest()

def github_put_file(branch,path,file):
    # Файл в новом собственном сегменте. Не редактирует main.
    payload={"message":"ML-05 verified full research output: "+file.name,"content":base64.b64encode(file.read_bytes()).decode("ascii"),"branch":branch}
    obj=github_json("PUT",API+"/contents/"+quote(path,safe="/"),json=payload,timeout=(30,900))
    return obj["commit"]["sha"]

def github_contents_sha(branch,path):
    target=API+"/contents/"+quote(path,safe="/")
    r=safe_request("GET",target,params={"ref":branch})
    v=r.json();r.close()
    if v.get("content"):
        return hashlib.sha256(base64.b64decode(v["content"])).hexdigest()
    r=safe_request("GET",target,params={"ref":branch},headers={"Accept":"application/vnd.github.raw+json"},stream=True)
    h=hashlib.sha256()
    for chunk in r.iter_content(2*1024*1024):
        if chunk:h.update(chunk)
    r.close();return h.hexdigest()

def publish_full():
    if not FULL_QA or CFG.MODE!="FULL":raise RuntimeError("PUBLICATION_BLOCKED: необходим реальный FULL LOCAL QA")
    source=github_json("GET",API+"/git/ref/heads/"+quote(CFG.INPUT_REF))
    sha=source["object"]["sha"]
    suffix=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    branch=CFG.TARGET_BRANCH_PREFIX+"-"+suffix
    # Отдельная новая ветка, никогда force-push.
    github_json("POST",API+"/git/refs",json={"ref":"refs/heads/"+branch,"sha":sha})
    root=CFG.SEGMENT.strip("/")+"/"
    commits={}
    for path in [NOTEBOOK,EXCEL]:
        commits[path.name]=github_put_file(branch,root+path.name,path)
    artifact_sha=commits[EXCEL.name]
    # Новый release tag привязан к изолированной ветке. Не перезаписываем существующую версию.
    release_tag=CFG.VERSION+"-"+suffix
    rel=github_json("POST",API+"/releases",json={"tag_name":release_tag,"target_commitish":branch,"name":"ML-05 · "+release_tag,"draft":False,"prerelease":True,"generate_release_notes":False,"body":"ML-05: validation-gated evidence for private research; SHA-256 in run_manifest.json"})
    data=EVIDENCE.read_bytes()
    upload_url="https://uploads.github.com/repos/"+CFG.REPO+"/releases/"+str(rel["id"])+"/assets"
    asset=github_json("POST",upload_url,params={"name":EVIDENCE.name},data=data,headers={"Content-Type":"application/zip"},timeout=(30,1800))
    link=asset["browser_download_url"]
    if github_binary(API+"/releases/assets/"+str(asset["id"]))!=sha256(EVIDENCE):raise RuntimeError("Проверка Release SHA-256 не пройдена")
    for file in [NOTEBOOK,EXCEL]:
        remote=github_contents_sha(branch,root+file.name)
        if remote!=sha256(file):raise RuntimeError("Опубликованный файл отличается: "+file.name)
    RUN_MANIFEST["git_publication"]="PASS_GITHUB"
    RUN_MANIFEST["publication_branch"]=branch
    RUN_MANIFEST["artifact_commit_sha"]=artifact_sha
    RUN_MANIFEST["release_tag"]=release_tag
    RUN_MANIFEST["evidence_release_asset_url"]=link
    RUN_MANIFEST["notebook_url"]="https://github.com/"+CFG.REPO+"/blob/"+branch+"/"+root+NOTEBOOK.name
    RUN_MANIFEST["excel_url"]="https://github.com/"+CFG.REPO+"/blob/"+branch+"/"+root+EXCEL.name
    RUN_MANIFEST["run_status"]="FULL_PASS"
    MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,default=str,indent=2),encoding="utf-8")
    commits[MANIFEST.name]=github_put_file(branch,root+MANIFEST.name,MANIFEST)
    if github_contents_sha(branch,root+MANIFEST.name)!=sha256(MANIFEST):raise RuntimeError("Опубликованный манифест не совпадает")
    return {"branch":branch,"tag":release_tag,"artifact_commit":artifact_sha,"manifest_commit":commits[MANIFEST.name]}

published=None
if CFG.MODE=="FULL" and FULL_QA and CFG.PUBLISH_AFTER_FULL_QA:
    try: published=publish_full()
    except Exception as error:
        RUN_MANIFEST["git_publication"]="PUBLICATION_FAIL"
        RUN_MANIFEST["run_status"]="PUBLICATION_FAIL"
        RUN_MANIFEST["publish_error_type"]=type(error).__name__
        MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,default=str,indent=2),encoding="utf-8")
        print("PUBLICATION_FAIL: публикация не подтверждена; детали в traceback.log")
        (OUT/"traceback.log").write_text(traceback.format_exc(),encoding="utf-8")
else:
    print("Публикация пропущена: SMOKE, PARTIAL или явно отключена в конфигурации")
BAR.update(1)
BAR.close()

# 15. Краткая сводка фактических проверок.
print("\nML-05 · "+RUN_MANIFEST["run_status"])
print("Данные:",len(od),"наблюдаемых строк СПО OD;",len(INPUT_HASHES),"проверенных скачанных исходников")
print("Модели:",len(METRICS),"записей оценки;",len(PRED),"тестовых прогнозов;",len(FIGURES),"реальных графиков")
for mod,s in STATUS.items():print(mod,s["status"],"—",s["reason"][:130])
if not metrics_df.empty:
    for _,r in metrics_df.loc[metrics_df.protocol=="ROLLING_TIME"].head(5).iterrows():print(r.module,r.model,"MAE=",round(float(r.MAE),4))
if published:print("Публикация:",published)
print("Локальные файлы:",NOTEBOOK,EXCEL,EVIDENCE,MANIFEST,sep="\n")